In [1]:
import sys

sys.path.append("..")
from src.model.geoclip import GeoCLIP
from src.model.g3 import G3
from src.utils import (
    build_index,
    add_record_to_index,
    save_index,
    get_device,
    read_index,
)
import polars as pl
import torch
from torch.nn import functional as F
from tqdm import tqdm
from src.model.backbones import load_backbone
import json

In [2]:
DEVICE = get_device()

In [3]:
geoclip_index, meta = read_index("../index/twostep-geoclip-flat-ip")
gps_index, _ = read_index("../index/twostep-geoclip-flat-ip", prefix="gps")
clip_index, clip_meta = read_index("../index/baseline-clip")
meta = meta["metadata"]

In [4]:
clip_backbone = load_backbone("clip", DEVICE)
geoclip_backbone = load_backbone("geoclip", DEVICE)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
with open(
    "/mnt/yokoyamalab-nas/gldv2-full/csv/queries/test_templated_queries.json", "r"
) as f:
    data = json.loads(f.read())["data"]
    data = [d for d in data if len(d["relevant_ids"]) >= 5]

In [ ]:
queries = [d["text"] for d in data]
# query = "a photo of tower"
query_embed = clip_backbone.encode_text(queries, 1).numpy()

encode text:   0%|          | 0/877 [00:00<?, ?it/s]

In [ ]:
for embed in tqdm(query_embed):
    _, indices = clip_index.search(embed, 10000)
    indices = indices.reshape(-1).tolist()
    docs = [meta[i] for i in indices]
    

KeyboardInterrupt: 

In [103]:
_, indices = clip_index.search(query_embed, 1000)
indices = indices.reshape(-1).tolist()
clip_out = [meta[i] for i in indices]

In [104]:
len([d for d in clip_out if d["category"] == "tower" and d["country"] == "United Kingdom"])

29

In [128]:
import os

paths = []
loc_pred = []

for doc in tqdm(clip_out):
    path = os.path.join("/mnt/yokoyamalab-nas/gldv2-full/", doc["src"], f"{doc["id"]}.jpg")
    pred, _ = geoclip_backbone.model.predict(path, 1)
    loc_pred.append({
        "id": doc["id"],
        "gps_pred": pred.view(-1).tolist()
    })

100%|██████████| 1000/1000 [02:30<00:00,  6.66it/s]


In [129]:
loc_pred

[{'id': '1a21864a7eb84db8',
  'gps_pred': [52.68548583984375, -1.8298629522323608]},
 {'id': '8b46b9ad5ade197d',
  'gps_pred': [52.0544548034668, -2.7164340019226074]},
 {'id': 'ef9aea812ff5abb6',
  'gps_pred': [53.12113952636719, -2.3212480545043945]},
 {'id': 'f2e15c16090f12b6',
  'gps_pred': [51.31564712524414, 9.408415794372559]},
 {'id': 'aaca26a5cfac70a1',
  'gps_pred': [52.20775604248047, 0.11790399998426437]},
 {'id': 'fb6b96b2fe906aff',
  'gps_pred': [51.909027099609375, -1.8381069898605347]},
 {'id': 'd27359dfbe2744b9',
  'gps_pred': [51.73707580566406, 11.023336410522461]},
 {'id': '54379d3f4a22c7ec',
  'gps_pred': [51.30180358886719, -2.3255820274353027]},
 {'id': '3a3139daf6f91410',
  'gps_pred': [52.68548583984375, -1.8298629522323608]},
 {'id': '9886494ea84ec4bf', 'gps_pred': [48.32389831542969, 8.96857738494873]},
 {'id': 'c695fbb21978ad0e',
  'gps_pred': [53.34194564819336, -6.309721946716309]},
 {'id': 'b54ec15cd4c69457',
  'gps_pred': [51.238441467285156, 13.41877460

In [114]:
import reverse_geocoder as rg

rg.search(pred.view(-1).tolist())

Loading formatted geocoded file...


[{'lat': '52.68154',
  'lon': '-1.82549',
  'name': 'Lichfield',
  'admin1': 'England',
  'admin2': 'Staffordshire',
  'cc': 'GB'}]